In [ ]:
# ==============================================================
#              YOUTUBE VIDEO DOWNLOADER
#                   INTERMEDIATE PROJECT
# ==============================================================
# GUI        : Tkinter
# Language   : Python
# Library    : PytubeFix
# Purpose    : Download permitted YouTube videos
#
# IMPORTANT:
# Use this application only for videos that you are legally
# permitted to download.
# ==============================================================


# --------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARY
# --------------------------------------------------------------
# This command installs pytubefix directly from Jupyter.
# The package provides access to YouTube video information
# and downloadable streams.

import sys
import subprocess

try:
    import pytubefix
except ImportError:
    print("Installing pytubefix...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "pytubefix"
    ])


# --------------------------------------------------------------
# 2. IMPORT LIBRARIES
# --------------------------------------------------------------

import os
import tkinter as tk
from tkinter import ttk, messagebox, filedialog

from pytubefix import YouTube


# --------------------------------------------------------------
# 3. GLOBAL VARIABLES
# --------------------------------------------------------------

# This variable stores the current YouTube object.
current_video = None

# This list stores available progressive streams.
available_streams = []


# --------------------------------------------------------------
# 4. MAIN APPLICATION WINDOW
# --------------------------------------------------------------

root = tk.Tk()

# Application title.
root.title("YouTube Video Downloader")

# Application size.
root.geometry("800x650")

# Prevent the user from resizing the window.
root.resizable(False, False)


# --------------------------------------------------------------
# 5. APPLICATION TITLE
# --------------------------------------------------------------

heading = tk.Label(
    root,
    text="YOUTUBE VIDEO DOWNLOADER",
    font=("Arial", 22, "bold")
)

heading.pack(pady=20)


# --------------------------------------------------------------
# 6. URL SECTION
# --------------------------------------------------------------

url_frame = tk.Frame(root)

url_frame.pack(pady=10)


tk.Label(
    url_frame,
    text="YouTube URL:",
    font=("Arial", 11, "bold")
).grid(
    row=0,
    column=0,
    padx=5,
    pady=5
)


# Entry box where the user enters the YouTube URL.
url_entry = tk.Entry(
    url_frame,
    width=65
)

url_entry.grid(
    row=0,
    column=1,
    padx=5,
    pady=5
)


# --------------------------------------------------------------
# 7. INFORMATION FRAME
# --------------------------------------------------------------

info_frame = tk.LabelFrame(
    root,
    text="Video Information",
    padx=15,
    pady=10
)

info_frame.pack(
    padx=20,
    pady=10,
    fill="x"
)


# Variable used to display video title.
title_var = tk.StringVar(value="Not loaded")

# Variable used to display channel.
channel_var = tk.StringVar(value="Not loaded")

# Variable used to display duration.
duration_var = tk.StringVar(value="Not loaded")


tk.Label(
    info_frame,
    text="Title:",
    font=("Arial", 10, "bold")
).grid(
    row=0,
    column=0,
    sticky="w",
    padx=5,
    pady=5
)


tk.Label(
    info_frame,
    textvariable=title_var,
    wraplength=620,
    justify="left"
).grid(
    row=0,
    column=1,
    sticky="w",
    padx=5,
    pady=5
)


tk.Label(
    info_frame,
    text="Channel:",
    font=("Arial", 10, "bold")
).grid(
    row=1,
    column=0,
    sticky="w",
    padx=5,
    pady=5
)


tk.Label(
    info_frame,
    textvariable=channel_var
).grid(
    row=1,
    column=1,
    sticky="w",
    padx=5,
    pady=5
)


tk.Label(
    info_frame,
    text="Duration:",
    font=("Arial", 10, "bold")
).grid(
    row=2,
    column=0,
    sticky="w",
    padx=5,
    pady=5
)


tk.Label(
    info_frame,
    textvariable=duration_var
).grid(
    row=2,
    column=1,
    sticky="w",
    padx=5,
    pady=5
)


# --------------------------------------------------------------
# 8. LOAD VIDEO FUNCTION
# --------------------------------------------------------------

def load_video():

    global current_video
    global available_streams

    # Get the URL entered by the user.
    url = url_entry.get().strip()

    # Check whether the URL is empty.
    if url == "":
        messagebox.showwarning(
            "Missing URL",
            "Please enter a YouTube video URL."
        )
        return

    try:

        # Create a YouTube object.
        current_video = YouTube(url)

        # Read basic information about the video.
        title = current_video.title
        channel = current_video.author

        # Convert duration from seconds to minutes and seconds.
        total_seconds = current_video.length

        minutes = total_seconds // 60
        seconds = total_seconds % 60

        duration = f"{minutes}:{seconds:02d}"

        # Display video information.
        title_var.set(title)
        channel_var.set(channel)
        duration_var.set(duration)

        # Clear the quality dropdown.
        quality_combo["values"] = ()

        # Get progressive MP4 streams.
        # Progressive streams contain both audio and video.
        available_streams = list(
            current_video.streams.filter(
                progressive=True,
                file_extension="mp4"
            )
        )

        # Remove duplicate resolutions while preserving
        # the best stream for each displayed resolution.
        stream_options = {}

        for stream in available_streams:

            resolution = stream.resolution

            if resolution is None:
                continue

            # Store the highest bitrate stream available
            # for each resolution.
            if resolution not in stream_options:
                stream_options[resolution] = stream
            else:
                existing = stream_options[resolution]

                if stream.bitrate and existing.bitrate:
                    if stream.bitrate > existing.bitrate:
                        stream_options[resolution] = stream

        # Sort resolutions numerically.
        sorted_resolutions = sorted(
            stream_options.keys(),
            key=lambda x: int(x.replace("p", "")),
            reverse=True
        )

        # Save only the selected streams.
        available_streams = [
            stream_options[resolution]
            for resolution in sorted_resolutions
        ]

        # Create readable quality labels.
        quality_options = [
            f"{stream.resolution} - {stream.mime_type}"
            for stream in available_streams
        ]

        # Add options to the dropdown.
        quality_combo["values"] = quality_options

        # Automatically select the first quality.
        if quality_options:
            quality_combo.current(0)

        messagebox.showinfo(
            "Video Loaded",
            "Video information loaded successfully."
        )

    except Exception as error:

        # Reset information if loading fails.
        current_video = None
        available_streams = []

        title_var.set("Not loaded")
        channel_var.set("Not loaded")
        duration_var.set("Not loaded")

        quality_combo["values"] = ()

        # Display the error.
        messagebox.showerror(
            "Error",
            f"Unable to load the video.\n\n{error}"
        )


# --------------------------------------------------------------
# 9. LOAD VIDEO BUTTON
# --------------------------------------------------------------

load_button = tk.Button(
    root,
    text="Load Video",
    command=load_video,
    width=20,
    font=("Arial", 10, "bold")
)

load_button.pack(pady=10)


# --------------------------------------------------------------
# 10. QUALITY SECTION
# --------------------------------------------------------------

quality_frame = tk.Frame(root)

quality_frame.pack(pady=15)


tk.Label(
    quality_frame,
    text="Select Quality:",
    font=("Arial", 10, "bold")
).grid(
    row=0,
    column=0,
    padx=5
)


# Dropdown for selecting video quality.
quality_combo = ttk.Combobox(
    quality_frame,
    state="readonly",
    width=35
)

quality_combo.grid(
    row=0,
    column=1,
    padx=5
)


# --------------------------------------------------------------
# 11. DOWNLOAD LOCATION
# --------------------------------------------------------------

folder_frame = tk.Frame(root)

folder_frame.pack(pady=10)


tk.Label(
    folder_frame,
    text="Save Location:",
    font=("Arial", 10, "bold")
).grid(
    row=0,
    column=0,
    padx=5
)


# Default download directory.
download_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads"
)

# Convert path to string variable.
folder_var = tk.StringVar(
    value=download_folder
)


# Display selected folder.
folder_entry = tk.Entry(
    folder_frame,
    textvariable=folder_var,
    width=55
)

folder_entry.grid(
    row=0,
    column=1,
    padx=5
)


# --------------------------------------------------------------
# 12. CHOOSE FOLDER FUNCTION
# --------------------------------------------------------------

def choose_folder():

    # Open a folder selection dialog.
    selected_folder = filedialog.askdirectory()

    # Update the folder when the user selects one.
    if selected_folder:
        folder_var.set(selected_folder)


# --------------------------------------------------------------
# 13. BROWSE BUTTON
# --------------------------------------------------------------

browse_button = tk.Button(
    folder_frame,
    text="Browse",
    command=choose_folder,
    width=10
)

browse_button.grid(
    row=0,
    column=2,
    padx=5
)


# --------------------------------------------------------------
# 14. PROGRESS BAR
# --------------------------------------------------------------

progress_frame = tk.Frame(root)

progress_frame.pack(
    pady=20
)


tk.Label(
    progress_frame,
    text="Download Progress:"
).pack(
    pady=5
)


progress_bar = ttk.Progressbar(
    progress_frame,
    orient="horizontal",
    length=500,
    mode="determinate"
)

progress_bar.pack(
    pady=5
)


# Variable for progress percentage.
progress_var = tk.StringVar(
    value="0%"
)


tk.Label(
    progress_frame,
    textvariable=progress_var,
    font=("Arial", 10, "bold")
).pack(
    pady=5
)


# --------------------------------------------------------------
# 15. DOWNLOAD PROGRESS CALLBACK
# --------------------------------------------------------------

def download_progress(
    stream,
    chunk,
    bytes_remaining
):

    # Calculate percentage completed.
    total_size = stream.filesize

    downloaded = total_size - bytes_remaining

    percentage = (
        downloaded / total_size
    ) * 100

    # Update progress bar.
    progress_bar["value"] = percentage

    # Update percentage text.
    progress_var.set(
        f"{percentage:.1f}%"
    )

    # Refresh the GUI so the progress is visible.
    root.update_idletasks()


# --------------------------------------------------------------
# 16. DOWNLOAD FUNCTION
# --------------------------------------------------------------

def download_video():

    global current_video

    # Check whether a video has been loaded.
    if current_video is None:
        messagebox.showwarning(
            "No Video",
            "Please load a YouTube video first."
        )
        return

    # Check whether at least one stream exists.
    if not available_streams:
        messagebox.showwarning(
            "No Streams",
            "No downloadable MP4 streams were found."
        )
        return

    # Get selected quality.
    selected_index = quality_combo.current()

    if selected_index == -1:
        messagebox.showwarning(
            "Quality Required",
            "Please select a video quality."
        )
        return

    # Get selected stream.
    selected_stream = available_streams[selected_index]

    # Get save directory.
    output_folder = folder_var.get().strip()

    # Check whether folder exists.
    if not os.path.isdir(output_folder):

        try:
            # Create the folder if it does not exist.
            os.makedirs(output_folder)

        except Exception as error:

            messagebox.showerror(
                "Folder Error",
                f"Unable to create the folder.\n\n{error}"
            )

            return

    # Reset progress.
    progress_bar["value"] = 0
    progress_var.set("0%")

    try:

        # Set the progress callback.
        current_video.register_on_progress_callback(
            download_progress
        )

        # Download the selected progressive stream.
        downloaded_file = selected_stream.download(
            output_path=output_folder
        )

        # Update progress to 100%.
        progress_bar["value"] = 100
        progress_var.set("100%")

        # Show the completed file path.
        messagebox.showinfo(
            "Download Complete",
            "Video downloaded successfully!\n\n"
            f"File:\n{downloaded_file}"
        )

    except Exception as error:

        # Display download errors.
        messagebox.showerror(
            "Download Error",
            f"The video could not be downloaded.\n\n{error}"
        )


# --------------------------------------------------------------
# 17. DOWNLOAD BUTTON
# --------------------------------------------------------------

download_button = tk.Button(
    root,
    text="DOWNLOAD VIDEO",
    command=download_video,
    width=25,
    height=2,
    font=("Arial", 11, "bold")
)

download_button.pack(
    pady=10
)


# --------------------------------------------------------------
# 18. CLEAR FUNCTION
# --------------------------------------------------------------

def clear_application():

    global current_video
    global available_streams

    # Reset all application variables.
    current_video = None
    available_streams = []

    # Clear URL.
    url_entry.delete(
        0,
        tk.END
    )

    # Reset video information.
    title_var.set("Not loaded")
    channel_var.set("Not loaded")
    duration_var.set("Not loaded")

    # Clear quality selection.
    quality_combo["values"] = ()

    # Reset progress.
    progress_bar["value"] = 0
    progress_var.set("0%")


# --------------------------------------------------------------
# 19. CLEAR BUTTON
# --------------------------------------------------------------

clear_button = tk.Button(
    root,
    text="Clear",
    command=clear_application,
    width=15
)

clear_button.pack(
    pady=5
)


# --------------------------------------------------------------
# 20. INFORMATION LABEL
# --------------------------------------------------------------

info_label = tk.Label(
    root,
    text=(
        "Enter a YouTube URL, load the video, select a quality, "
        "choose a folder, and click Download."
    ),
    font=("Arial", 9)
)

info_label.pack(
    pady=15
)


# --------------------------------------------------------------
# 21. START APPLICATION
# --------------------------------------------------------------

# Run the Tkinter event loop.
root.mainloop()


# ==============================================================
#                       END OF PROGRAM
# ==============================================================